## Installations

In [ ]:
%pip install matplotlib
from pathlib import Path
from time import perf_counter
import requests
import pandas as pd
import kagglehub
import os
import numpy as np 
from ollama import chat

OLLAMA_MODEL= "llama3.2:3b"
try:
    response = requests.get("http://localhost:11434")
    print("Ollama is running!")
except:
    print("Ollama is NOT running.")
    print("Start Ollama before continuing.")

## Load dataset/put in pandas DF

In [ ]:
#Testing Variables
TEMPERATURE = 0.2

EVALUATION_SCALE_MIN = 1
EVALUATION_SCALE_MAX = 5

In [ ]:
#Download dataset
path = kagglehub.dataset_download("sharmaabhi04/100k-movies-dataset")

#convert to CSV File
file_path = "100k_Movies_dataset.csv"

#load CSV file into a pandas DataFrame
movies_df = pd.read_csv(os.path.join(path, file_path))

In [ ]:
#explore data
print(movies_df.shape)
movies_df.columns.tolist()
movies_df.info()
movies_df.describe()
movies_df.head()

# RAG

In [ ]:
# # Load an embedding model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

In [ ]:
#word based chunking
def split_text_into_chunks(
    text,
    chunk_size=30,
    overlap=5
):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
# Chunking
all_chunks = []
all_metadata = []

chunk_id = 0  # Global chunk counter

for index in movies_df.itertuples(index=True):

    chunks = split_text_into_chunks(
        index.title,
        chunk_size=100,
        overlap=20
    )

    for chunk in chunks:

        all_chunks.append(chunk)

        all_metadata.append({
            "runtime": index.runtime,
            "genre": index.genre,
            "ratings": index.ratings,
            "director": index.director,
            "cast": index.cast,
            "Description": index.Description,
            "released_year": index.released_year,
            "chunk_id": chunk_id
        })

        chunk_id += 1  # Increment after each chunk

print("Total CSV chunks:", len(all_chunks))


## Functions

In [ ]:
# retrieval function
def retrieve_context(
    question,
    number_of_results=3
):
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
    query_embeddings=[question_embedding],
    n_results=number_of_results
    )
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    return documents, metadatas, distances

In [ ]:

# function to generate answer
def generate_rag_answer(
    question,
    number_of_results=3
):
    print("user_question:", question)
    
    documents, metadatas, distances = retrieve_context(
        question,
        number_of_results
    )

    context = "\n\n".join(documents)

    prompt = f"""
You are a movie recommendation assistant.

Answer the question using only the context below.

If the answer is not available in the context, say:
"I do not have enough verified information to answer
that question."

Context:
{context}

Question:
{question}

Answer:
"""

    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    answer = response["message"]["content"]

    return {
        "question": question,
        "answer": answer,
        "documents": documents,
        "metadata": metadatas,
        "distances": distances
    }

In [ ]:
def generate_response(
    prompt,
    temperature=TEMPERATURE
):
    start_time = perf_counter()

    response = chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": temperature
        }
    )

    elapsed_time = perf_counter() - start_time

    return {
        "text": response["message"]["content"],
        "response_time": elapsed_time
    }

## Vectors in ChromaDB 

In [ ]:
# ## ChromaDB
# creating db client
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)
print("ChromaDB client created.")

In [ ]:
# create collection
collection_name = "movie_information"
try:
    chroma_client.delete_collection(
        collection_name
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name
)

# print("Collection created:", collection_name)

In [ ]:
# chunk embedding
chunk_embeddings = embedding_model.encode(
    all_chunks
).tolist()

print(
    len(chunk_embeddings),
    "chunk embeddings created."
)

In [ ]:
# IDs and metadata creation
chunk_ids = [
    f"chunk_{index}"
    for index in range(len(all_chunks))
]

metadata = [
    {
        "source": "movie_dataset",
        "chunk_number": index
    }
    for index in range(len(all_chunks))
]

print("Number of IDs:", len(chunk_ids))
print("Number of metadata:", len(metadata))

print(chunk_ids[:5])
print(metadata[:5])

In [ ]:
#Dataset exceeds ChromaDB's batch limit
#ChatGPT recommended to add documents in batches of 5000 or less
batch_size = 5000

for start in range(0, len(all_chunks), batch_size):

    end = start + batch_size

    collection.add(
        ids=chunk_ids,
        documents=all_chunks,
        metadatas=all_metadata
    )

    print(f"Added documents {start} to {end}")

print("Documents stored in ChromaDB.")
print("Collection size:", collection.count())


# Semantic Retrival

In [ ]:
question = "Give a list of comedy movies realsed in 2020 with rating 8 or higher."
print(question)
# ques embedding generation
question_embedding = embedding_model.encode(
    question
).tolist()

print("Query embedding created.")    




In [ ]:
# top 3 chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print(results)

In [ ]:
# display documents retrieved
retrieved_documents = results["documents"][0]
retrieved_distances = results["distances"][0]
retrieved_metadata = results["metadatas"][0]

for index, document in enumerate(
    retrieved_documents
):
    print(f"Result {index + 1}")
    print("Document:", document)
    print("Distance:", retrieved_distances[index])
    print("Metadata:", retrieved_metadata[index])
    print("-" * 60)

## Build Rag Prompt

In [ ]:
# combine all the context that was retrieved
context = "\n\n".join(
    retrieved_documents
)

print(context)

In [ ]:
# grounded prompt creation
prompt = f"""
You are a Movie Recommendation assistant.

Answer the question using only the provided context.

If the context does not contain enough information,
say:

"I do not have enough verified information to answer
that question."

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)

In [ ]:
#Rag Answer
OLLAMA_MODEL= "llama3.2:3b"

response = chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    options={
        "temperature": 0.2
    }
)

answer = response["message"]["content"]

print(answer)

# Tests

## System Testing

In [ ]:
system_prompt = """
You are our AI movie recommendation assistant.
Your job is to recommend movies based on user preferences.
You should:
- Ask questions when the user gives unclear preferences.
- Explain why each recommendation matches the user's interests.
- Never invent movies, actors, ratings, or availability.
- Admit when you do not know something.
"""
print("====SYSTEM PROMPT=====")
print(system_prompt)

In [ ]:
print("\n===WEAK PROMPTS===\n")
weak_prompts = [
    "Recommend me some movies.",
    "I need a good movie.",
    "What movie should I watch?",
    "Give me something interesting.",
    "Tell me the best movie."
]
print("1. ", weak_prompts[0])
print("2. ", weak_prompts[1])
print("3. ", weak_prompts[2])
print("4. ", weak_prompts[3])
print("5. ", weak_prompts[4])
print("\n===STRONG PROMPTS===\n")

strong_prompts = [
    "Recommend 5 science fiction movies similar to Interstellar. I like space exploration and emotional stories.",

    "I want a family-friendly comedy movie that teenagers and adults can enjoy. Avoid R-rated movies.",

    "I enjoyed Knight and the Seven Kingdoms. Recommend a series with similar themes and storytelling.",

    "Recommend horror movies that focus on suspense instead of gore.",

    "I only have 90 minutes. Recommend highly-rated movies under that runtime."

]
print("1. ", strong_prompts[0])
print("2. ", strong_prompts[1])
print("3. ", strong_prompts[2])
print("4. ", strong_prompts[3])
print("5. ", strong_prompts[4])

## Functional Testing

In [ ]:
# testing complete RAG
question = "Recommend 3 scary movies above 4 stars"
result = generate_rag_answer(question, 3)
print("Question:")

print(result["question"])

print("\nAnswer:")
print(result["answer"])

## RAG Testing

In [ ]:
import ollama
import requests
import subprocess
import time

messages=[
    {
        "role":"user",
        "content":"Recommend movies 3 stars and up."
    }
]

# Call ollama.chat with stream=False to get the full response as a dictionary
response = ollama.chat(
    model="llama3.2:3b",
    messages=messages,
    stream=False # Set stream to False to get a dictionary response
)

# Now, 'response' is a dictionary, and you can access its elements
print(response["message"]["content"])

In [ ]:
# Few shot prompting
prompt = """
Movie Based Reccomendations

Example 1

Ex: User asks: I want something funny to cheer me up
Assistant: Here are some great comedy movies:

Paddington 2, Chef, and the Princess Bride


Example 2

User asks: I want something emotional
Assistant: Ok, here are some emotional movies you might like
Manchester by the sea
About Time
Interstellar


Example 3

Recommend based on favorite Movies
Ex. User asks: I loved Spider-Man: No way Home, Avengers: End Game, and
Guardians of the Galaxy
Assistant: You seem to enjoy superhero movies. Based on that, I recommend:
Shang-Chi and the Legend of the Ten Rings
Thor: Ragnarok
The Suicide Squad

"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# System Prompt
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"system",
            "content":"You are a parent of a young child, what movies would you recommend?"
        },
        {
            "role":"user",
            "content":"Recommend Movies."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# One-shot prompting
prompt = """
Example

Sentence:
The movie was fantastic.

Sentiment:
Positive

Sentence:
The assignment was confusing.

Sentiment:
"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

## Prompt Testing

In [ ]:
print("\n===ZERO-SHOT PROMPTS===\n")

zero_shot_prompts = [
    "Recommend a movie for a rainy day.",
    "What's a good movie to watch with my parents?",
    "Suggest a movie similar to Inception.",
    "Give me a movie recommendation for someone who likes slow-burn dramas.",
    "What should I watch if I only have 90 minutes free?"
]

print("1. ", zero_shot_prompts[0])
print("2. ", zero_shot_prompts[1])
print("3. ", zero_shot_prompts[2])
print("4. ", zero_shot_prompts[3])
print("5. ", zero_shot_prompts[4])

In [ ]:
print("\n===ZERO-SHOT RESPONSES===\n")

for i, prompt in enumerate(zero_shot_prompts, start=1):
    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.7
        }
    )
    print(f"{i}. PROMPT: {prompt}")
    print(f"   RESPONSE: {response['message']['content']}")
    print("\n")

## Hallucination Testing

In [ ]:
print("\n===HALLUCINATION PROMPTS===\n")
hallucination_prompts = [
    "Tell me about the movie The Last Ocean Planet starring Chris Evans.",

    "Why did Titanic 2: Return of the Ocean win Best Picture?",

    "Recommend movies directed by Christopher Nolan before 1900.",

    "What superhero movies did Emily Watson Jr. star in?",

    "Is Avatar 5 currently available on Netflix?"
]
print("1. ", hallucination_prompts[0])
print("2. ", hallucination_prompts[1])
print("3. ", hallucination_prompts[2])
print("4. ", hallucination_prompts[3])
print("5. ", hallucination_prompts[4])

In [ ]:
# safe prompt
def create_grounded_prompt(
    question,
    context
):
    return f"""
You are a movie recommendation assistant.

Answer only using the supplied context.

If the context does not contain the answer,
respond exactly with:

"I do not have enough infor to answer this questions"

Do not guess.
Do not invent information.
Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
# test
unsupported_question = (
    "What is the instructor's private phone number?"
)

safe_prompt = create_grounded_prompt(
    unsupported_question,
    context
)

unsupported_result = generate_response(
    safe_prompt
)

print(unsupported_result["text"])